# Pronunciation Scoring — Inference Notebook

Run end-to-end pronunciation scoring on any audio file (WAV / MP3 / FLAC / …)  
or a raw NumPy waveform array.

**Pipeline:**
```
Raw audio  →  HuBERT (layer-weighted)  →  Cross-attention fusion (+ Qwen3 text)  →  Transformer block  →  MLP  →  5-dim scores
```

## 1 · Imports

In [1]:
# All heavy imports are encapsulated in notebook_infer/
from notebook_infer import ScoreConfig, load_predictor, score_file, score_array
from notebook_infer.display import show_scores, show_words, show_waveform, play_audio

d:\spring_2026_project\slp\pronunciation-scoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 · Configuration

In [2]:
import torch

cfg = ScoreConfig(
    checkpoint   = "ckpt_hubert_multitask/best_2.pt",  # set to None for untrained random weights
    device       = "cuda" if torch.cuda.is_available() else "cpu",
    whisper_size = "small",                           # ASR model size: tiny / small / medium
    language     = "en",                              # ISO-639-1 language code
    text_model   = "Qwen/Qwen3-Embedding-0.6B",       # text embedding model
)
print(cfg)

ScoreConfig(checkpoint='ckpt_hubert_multitask/best_2.pt', device='cpu', hubert_name='facebook/hubert-base-ls960', whisper_size='small', language='en', text_model='Qwen/Qwen3-Embedding-0.6B')


## 3 · Load model

This downloads HuBERT and Qwen3-Embedding on first run (~700 MB total).  
Subsequent runs load from the HuggingFace cache.

In [3]:
predictor = load_predictor(cfg)
print("Model ready.")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2168.14it/s]


Model ready.


---
## 4A · Score from a file path

In [4]:
# WAV_PATH = "D:/spring_2026_project/slp/pronunciation-scoring/audio/02_native.wav"   # ← change to your audio file
WAV_PATH = "./audio/learner/07_learner.wav"
result = score_file(WAV_PATH, predictor, cfg)

---
## 4B · Score from a raw NumPy array

Use this if you already have the waveform in memory (e.g. from `soundfile.read` or `librosa.load`).

In [5]:
import numpy as np
import soundfile as sf

# Load raw audio into a NumPy array (replace with your own source)
audio_np, sample_rate = sf.read(WAV_PATH, dtype="float32", always_2d=False)
print(f"Audio shape: {audio_np.shape}  |  Sample rate: {sample_rate} Hz")

# Score directly from the array
result = score_array(audio_np, sample_rate, predictor, cfg)

Audio shape: (73387,)  |  Sample rate: 16000 Hz


---
## 5 · Display results

In [6]:
show_scores(result)

Dimension,Score / 10
Total,6.04
Accuracy,5.16
Fluency,5.36
Prosodic,4.50
Completeness,4.56


In [ ]:
show_words(result)

In [ ]:
show_waveform(WAV_PATH, result)

In [ ]:
play_audio(WAV_PATH)

---
## 6 · Raw JSON output

In [ ]:
import json
print(json.dumps(result, indent=2, ensure_ascii=False))

---
## 7 · Batch scoring

Score multiple files and collect results into a DataFrame.

In [ ]:
# import glob
# import pandas as pd

# audio_files = glob.glob("audio/**/*.wav", recursive=True)
# print(f"Found {len(audio_files)} audio files.")

# rows = []
# for path in audio_files:
#     r = score_file(path, predictor, cfg)
#     rows.append({
#         "file":         path,
#         "transcript":   r.get("text"),
#         "total":        r.get("total"),
#         "accuracy":     r.get("accuracy"),
#         "fluency":      r.get("fluency"),
#         "prosodic":     r.get("prosodic"),
#         "completeness": r.get("completeness"),
#     })

# df = pd.DataFrame(rows)
# df